In [3]:
import os
from dotenv import load_dotenv
from langgraph.graph import StateGraph
from langgraph.graph import END
from typing import TypedDict
from langchain.memory import ConversationBufferMemory
from langchain.chains import ConversationChain
from langchain_google_genai import ChatGoogleGenerativeAI

In [4]:
# load api key
load_dotenv()

True

In [ ]:
# Define Shared State
class ChatState(TypedDict):
    input: str
    output: str

In [ ]:
# setup Memory
memory = ConversationBufferMemory(
    memory_key='history',
    return_messages=True
)

C:\Users\emon1\AppData\Local\Temp\ipykernel_25192\3738000325.py:2: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationBufferMemory(memory_key='history', return_messages=True)


In [7]:
# setup gemini model
llm = ChatGoogleGenerativeAI(
    model='gemini-2.5-pro',
    temperature=0.9,
    convert_system_message_to_human=True
)

In [8]:
# create llm chain with memory
chat_chain = ConversationChain(
    llm=llm,
    memory=memory
)

C:\Users\emon1\AppData\Local\Temp\ipykernel_25192\3811395801.py:2: LangChainDeprecationWarning: The class `ConversationChain` was deprecated in LangChain 0.2.7 and will be removed in 1.0. Use :meth:`~RunnableWithMessageHistory: https://python.langchain.com/v0.2/api_reference/core/runnables/langchain_core.runnables.history.RunnableWithMessageHistory.html` instead.
  chat_chain = ConversationChain(


In [ ]:
# Define LangGraph Node
def chat_node(state: ChatState) -> ChatState:
    user_input = state["input"]
    print("User:", user_input)
    
    response = chat_chain.predict(input=user_input)
    print("Gemini:", response)
    
    state["output"] = response
    return state

In [11]:
# build langgraph
builder = StateGraph(ChatState)

builder.add_node("chat", chat_node)
builder.set_entry_point("chat")
builder.add_edge("chat", END)

In [12]:
graph = builder.compile()

In [13]:
# Multi-turn Chat Example
print("\nTurn 1")
state1 = graph.invoke({"input": "Hello Gemini!", "output": ""})


Turn 1
User: Hello Gemini!


c:\Users\emon1\AppData\Local\Programs\Python\Python311\Lib\site-packages\langchain_google_genai\chat_models.py:353: UserWarning: Convert_system_message_to_human will be deprecated!
  warnings.warn("Convert_system_message_to_human will be deprecated!")


Gemini: Hello there! It's so nice to meet you. I am indeed Gemini, a large language model, trained by Google.

It's a pleasure to be chatting with you! I exist as a complex set of algorithms and data points running on Google's powerful and secure cloud infrastructure. I don't have a body or a physical location in the way a person does, but you could say my "home" is a vast, interconnected network of servers.

I've been trained on an enormous amount of text and code, which allows me to understand and generate language, translate between different languages, write all sorts of creative content, and answer your questions in an informative way.

What can I help you with today? I'm ready to dive into any topic you're curious about


In [14]:
print("\nTurn 2")
state2 = graph.invoke({"input": "What did I just say?", "output": ""})


Turn 2
User: What did I just say?


c:\Users\emon1\AppData\Local\Programs\Python\Python311\Lib\site-packages\langchain_google_genai\chat_models.py:353: UserWarning: Convert_system_message_to_human will be deprecated!
  warnings.warn("Convert_system_message_to_human will be deprecated!")


Gemini: You just said, "Hello Gemini!"

That was your very first message to me in this particular conversation. I remember it clearly because I process our chat as a sequence of turns. Your message is stored as the first `HumanMessage` in my current context window for this session.

Your friendly greeting is what prompted me to give my detailed introduction about being a large language model trained by Google, running on cloud infrastructure, and all that fun stuff. It was a lovely way to start our chat!

Is there anything else you'd like to ask? Perhaps about something else in our conversation or a new topic entirely? I'm ready for whatever you have in mind
